In [11]:
"""
Phase 3: Quantization for FPGA Implementation
Author: ECG-ID Hardware Acceleration Project
Purpose: Convert floating-point weights to fixed-point (Q7.8 format)
"""

import numpy as np
import os

# ============================================================================
# CONFIGURATION
# ============================================================================
MODEL_DIR = "model_outputs"  # The folder visible in your sidebar
DATA_DIR = "processed_data"   # The folder containing X_test and y_test

FRACTIONAL_BITS = 8  # Q7.8 format (8 bits for fractional part)
SCALE_FACTOR = 2 ** FRACTIONAL_BITS  # 256
TOTAL_BITS = 16  # 1 sign + 7 integer + 8 fractional
MIN_VALUE = -(2 ** (TOTAL_BITS - 1))  # -32768
MAX_VALUE = (2 ** (TOTAL_BITS - 1)) - 1  # 32767

# File paths (adjust these to match your actual file locations)
WEIGHTS_FILE = os.path.join(MODEL_DIR, "w_hidden.npy")
BIAS_HIDDEN_FILE = os.path.join(MODEL_DIR, "b_hidden.npy")
WEIGHTS_OUTPUT_FILE = os.path.join(MODEL_DIR, "w_output.npy")
BIAS_OUTPUT_FILE = os.path.join(MODEL_DIR, "b_output.npy")

# Output directory
OUTPUT_DIR = "quantized_outputs"

# ============================================================================
# STEP A: QUANTIZATION FUNCTIONS
# ============================================================================

def quantize_array(float_array, name="array"):
    """
    Quantize an entire numpy array using vectorized NumPy operations.
    """
    print(f"\nQuantizing {name}...")
    
    # 1. Scale everything at once
    scaled = float_array * SCALE_FACTOR
    
    # 2. Round everything using NumPy's rounding (handles the error you saw)
    rounded = np.round(scaled)
    
    # 3. Clamp values to prevent 16-bit overflow
    quantized = np.clip(rounded, MIN_VALUE, MAX_VALUE).astype(np.int16)
    
    print(f"  Original range: [{float_array.min():.6f}, {float_array.max():.6f}]")
    print(f"  Quantized range: [{quantized.min()}, {quantized.max()}]")
    
    return quantized


def quantize_array(float_array, name="array"):
    """
    Quantize an entire numpy array
    
    Args:
        float_array: NumPy array of floating-point values
        name: Name for logging purposes
    
    Returns:
        Quantized integer array (int16)
    """
    print(f"\nQuantizing {name}...")
    print(f"  Original shape: {float_array.shape}")
    print(f"  Original range: [{float_array.min():.6f}, {float_array.max():.6f}]")
    
    # Apply quantization
    quantized = np.vectorize(quantize_value)(float_array).astype(np.int16)
    
    print(f"  Quantized range: [{quantized.min()}, {quantized.max()}]")
    print(f"  Sample original values: {float_array.flatten()[:3]}")
    print(f"  Sample quantized values: {quantized.flatten()[:3]}")
    
    return quantized


def dequantize_value(int_value):
    """
    Convert Q7.8 integer back to floating-point for verification
    
    Args:
        int_value: Quantized integer
    
    Returns:
        Approximate floating-point value
    """
    return int_value / SCALE_FACTOR


# ============================================================================
# STEP B: INTEGER INFERENCE TEST
# ============================================================================

def relu_fixed(x):
    """ReLU activation for fixed-point integers"""
    return np.maximum(0, x)


def sigmoid_fixed(x):
    """
    Approximate sigmoid for fixed-point (scaled output)
    Returns value in range [0, 256] representing [0.0, 1.0]
    """
    # Dequantize for sigmoid calculation
    x_float = x / SCALE_FACTOR
    sigmoid_float = 1 / (1 + np.exp(-np.clip(x_float, -10, 10)))
    return int(round(sigmoid_float * SCALE_FACTOR))


def mlp_inference_fixed(input_sample, W1_int, b1_int, W2_int, b2_int):
    """
    Perform inference using quantized integer weights
    
    Args:
        input_sample: Input ECG window (100 values, already scaled by 256)
        W1_int, b1_int, W2_int, b2_int: Quantized weights and biases
    
    Returns:
        Output probability (0-256 scale)
    """
    # Hidden layer: z1 = W1^T * x + b1
    z1 = np.dot(input_sample, W1_int) + (b1_int * SCALE_FACTOR)  # Keep in Q7.8
    
    # ReLU activation
    a1 = relu_fixed(z1)
    
    # Scale down to prevent overflow (divide by 256)
    a1 = a1 // SCALE_FACTOR
    
    # Output layer: z2 = W2^T * a1 + b2
    z2 = np.dot(a1, W2_int.flatten()) + (b2_int * SCALE_FACTOR)
    
    # Sigmoid activation
    output = sigmoid_fixed(z2)
    
    return output


def test_quantized_model(X_test, y_test, W1_int, b1_int, W2_int, b2_int):
    """
    Test the quantized model accuracy
    
    Returns:
        Accuracy percentage
    """
    print("\n" + "="*70)
    print("STEP B: TESTING INTEGER BRAIN")
    print("="*70)
    
    correct = 0
    total = len(X_test)
    
    # Scale input data to integer domain
    X_test_int = (X_test * SCALE_FACTOR).astype(np.int16)
    
    for i, (x_sample, y_true) in enumerate(zip(X_test_int, y_test)):
        # Integer inference
        output_int = mlp_inference_fixed(x_sample, W1_int, b1_int, W2_int, b2_int)
        
        # Convert to probability (0-1 range)
        prob = output_int / SCALE_FACTOR
        y_pred = 1 if prob >= 0.5 else 0
        
        if y_pred == y_true:
            correct += 1
        
        # Show progress
        if (i + 1) % 1000 == 0:
            print(f"  Tested {i+1}/{total} samples...")
    
    accuracy = (correct / total) * 100
    print(f"\n✓ Quantized Model Accuracy: {accuracy:.2f}%")
    
    return accuracy


# ============================================================================
# STEP C: VERILOG HEADER GENERATION
# ============================================================================

def generate_verilog_header(W1_int, b1_int, W2_int, b2_int, output_file):
    """
    Generate a Verilog header file (.vh) with all quantized parameters
    """
    print(f"\n{'='*70}")
    print("STEP C: GENERATING VERILOG HEADER")
    print(f"{'='*70}")
    
    with open(output_file, 'w') as f:
        f.write("// ============================================\n")
        f.write("// Quantized Neural Network Parameters\n")
        f.write("// Format: Q7.8 (16-bit signed integers)\n")
        f.write("// Scale Factor: 256\n")
        f.write(f"// Total Parameters: {W1_int.size + b1_int.size + W2_int.size + b2_int.size}\n")
        f.write("// ============================================\n\n")
        
        # Hidden layer weights (100x32)
        f.write("// Hidden Layer Weights (W1): 100 inputs x 32 neurons\n")
        for i in range(W1_int.shape[0]):
            for j in range(W1_int.shape[1]):
                f.write(f"parameter signed [15:0] W1_{i}_{j} = 16'sd{W1_int[i,j]};\n")
        f.write("\n")
        
        # Hidden layer biases (32)
        f.write("// Hidden Layer Biases (b1): 32 neurons\n")
        for i in range(len(b1_int)):
            f.write(f"parameter signed [15:0] b1_{i} = 16'sd{b1_int[i]};\n")
        f.write("\n")
        
        # Output layer weights (32x1)
        f.write("// Output Layer Weights (W2): 32 inputs x 1 output\n")
        for i in range(W2_int.shape[0]):
            f.write(f"parameter signed [15:0] W2_{i} = 16'sd{W2_int[i,0]};\n")
        f.write("\n")
        
        # Output layer bias (1)
        f.write("// Output Layer Bias (b2): 1 output\n")
        f.write(f"parameter signed [15:0] b2_0 = 16'sd{b2_int[0]};\n")
    
    print(f"✓ Verilog header saved to: {output_file}")
    
    # Generate statistics file
    stats_file = output_file.replace('.vh', '_stats.txt')
    with open(stats_file, 'w') as f:
        f.write("Quantization Statistics\n")
        f.write("="*50 + "\n\n")
        f.write(f"Format: Q7.8 (16-bit signed)\n")
        f.write(f"Scale Factor: {SCALE_FACTOR}\n")
        f.write(f"Representable Range: [{MIN_VALUE/SCALE_FACTOR:.4f}, {MAX_VALUE/SCALE_FACTOR:.4f}]\n\n")
        f.write(f"W1 range: [{W1_int.min()}, {W1_int.max()}]\n")
        f.write(f"b1 range: [{b1_int.min()}, {b1_int.max()}]\n")
        f.write(f"W2 range: [{W2_int.min()}, {W2_int.max()}]\n")
        f.write(f"b2 range: [{b2_int.min()}, {b2_int.max()}]\n")
    
    print(f"✓ Statistics saved to: {stats_file}")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    print("="*70)
    print("PHASE 3: QUANTIZATION FOR FPGA IMPLEMENTATION")
    print("="*70)
    
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # ========================================================================
    # STEP A: LOAD AND QUANTIZE WEIGHTS
    # ========================================================================
    print("\n" + "="*70)
    print("STEP A: LOADING AND QUANTIZING WEIGHTS")
    print("="*70)
    
    try:
        # Load floating-point weights
        W1_float = np.load(WEIGHTS_FILE)
        b1_float = np.load(BIAS_HIDDEN_FILE)
        W2_float = np.load(WEIGHTS_OUTPUT_FILE)
        b2_float = np.load(BIAS_OUTPUT_FILE)
        
        print(f"\n✓ Loaded all weight files successfully")
        
        # Quantize all parameters
        W1_int = quantize_array(W1_float, "W1 (Hidden Weights)")
        b1_int = quantize_array(b1_float, "b1 (Hidden Biases)")
        W2_int = quantize_array(W2_float, "W2 (Output Weights)")
        b2_int = quantize_array(b2_float, "b2 (Output Bias)")
        
        # Save quantized weights
        np.save(f"{OUTPUT_DIR}/W1_quantized.npy", W1_int)
        np.save(f"{OUTPUT_DIR}/b1_quantized.npy", b1_int)
        np.save(f"{OUTPUT_DIR}/W2_quantized.npy", W2_int)
        np.save(f"{OUTPUT_DIR}/b2_quantized.npy", b2_int)
        
        print(f"\n✓ Quantized weights saved to '{OUTPUT_DIR}/' directory")
        
    except FileNotFoundError as e:
        print(f"\n✗ ERROR: Could not find weight files!")
        print(f"  Please ensure the following files exist:")
        print(f"    - {WEIGHTS_FILE}")
        print(f"    - {BIAS_HIDDEN_FILE}")
        print(f"    - {WEIGHTS_OUTPUT_FILE}")
        print(f"    - {BIAS_OUTPUT_FILE}")
        return
    
    # ========================================================================
    # STEP B: TEST QUANTIZED MODEL (Optional - requires test data)
    # ========================================================================
    test_data_file = os.path.join(DATA_DIR, "X_test.npy")
    test_labels_file = os.path.join(DATA_DIR, "y_test.npy")
    
    if os.path.exists(test_data_file) and os.path.exists(test_labels_file):
        try:
            X_test = np.load(test_data_file)
            y_test = np.load(test_labels_file)
            
            accuracy = test_quantized_model(X_test, y_test, W1_int, b1_int, W2_int, b2_int)
            
            if accuracy > 95:
                print("  ✓ EXCELLENT! Quantization preserved accuracy!")
            elif accuracy > 90:
                print("  ⚠ GOOD: Minor accuracy loss, acceptable for hardware")
            else:
                print("  ✗ WARNING: Significant accuracy drop. Consider increasing bit precision.")
        except Exception as e:
            print(f"\n⚠ Could not test quantized model: {e}")
            print("  Skipping accuracy test...")
    else:
        print(f"\n⚠ Test data not found. Skipping Step B (accuracy test)")
        print(f"  To run accuracy test, ensure these files exist:")
        print(f"    - {test_data_file}")
        print(f"    - {test_labels_file}")
    
    # ========================================================================
    # STEP C: GENERATE VERILOG HEADER
    # ========================================================================
    verilog_file = f"{OUTPUT_DIR}/nn_parameters.vh"
    generate_verilog_header(W1_int, b1_int, W2_int, b2_int, verilog_file)
    
    # ========================================================================
    # FINAL SUMMARY
# ========================================================================
    print("\n" + "="*70)
    print("PHASE 3 COMPLETE!")
    print("="*70)
    print(f"\n📁 Output Files:")
    print(f"   1. {OUTPUT_DIR}/W1_quantized.npy")
    print(f"   2. {OUTPUT_DIR}/b1_quantized.npy")
    print(f"   3. {OUTPUT_DIR}/W2_quantized.npy")
    print(f"   4. {OUTPUT_DIR}/b2_quantized.npy")
    print(f"   5. {verilog_file}")
    print(f"   6. {verilog_file.replace('.vh', '_stats.txt')}")
    
    print(f"\n🚀 Next Steps:")
    print(f"   → Open '{verilog_file}' to see your Verilog parameters")
    print(f"   → Ready to implement in Verilog (Phase 4)!")
    print(f"\n" + "="*70)


if __name__ == "__main__":
    main()

PHASE 3: QUANTIZATION FOR FPGA IMPLEMENTATION

STEP A: LOADING AND QUANTIZING WEIGHTS

✓ Loaded all weight files successfully

Quantizing W1 (Hidden Weights)...
  Original shape: (100, 32)
  Original range: [-1.740968, 0.979690]
  Quantized range: [-446, 251]
  Sample original values: [0.195205   0.27788666 0.04792117]
  Sample quantized values: [50 71 12]

Quantizing b1 (Hidden Biases)...
  Original shape: (32,)
  Original range: [-0.184736, 0.317371]
  Quantized range: [-47, 81]
  Sample original values: [0.01168987 0.17153496 0.15977462]
  Sample quantized values: [ 3 44 41]

Quantizing W2 (Output Weights)...
  Original shape: (32, 1)
  Original range: [-1.224844, 0.871251]
  Quantized range: [-314, 223]
  Sample original values: [-0.8017551  -0.5448001   0.77141285]
  Sample quantized values: [-205 -139  197]

Quantizing b2 (Output Bias)...
  Original shape: (1,)
  Original range: [-0.057220, -0.057220]
  Quantized range: [-15, -15]
  Sample original values: [-0.05722031]
  Sample 

In [12]:
import numpy as np
import os

# --- 1. SETTINGS ---
QUANT_DIR = "quantized_outputs"
DATA_DIR = "processed_data"
SCALE_FACTOR = 256  # Q7.8

def sigmoid_fixed(x):
    # Mimics hardware lookup table / approximation
    x_float = x / (SCALE_FACTOR * SCALE_FACTOR) # Account for the multiplication shift
    return 1 / (1 + np.exp(-np.clip(x_float, -10, 10)))

# --- 2. LOAD EVERYTHING ---
print("📂 Loading data and quantized weights...")
W1 = np.load(os.path.join(QUANT_DIR, "W1_quantized.npy"))
b1 = np.load(os.path.join(QUANT_DIR, "b1_quantized.npy"))
W2 = np.load(os.path.join(QUANT_DIR, "W2_quantized.npy"))
b2 = np.load(os.path.join(QUANT_DIR, "b2_quantized.npy"))

X_test = np.load(os.path.join(DATA_DIR, "X_test.npy"))
y_test = np.load(os.path.join(DATA_DIR, "y_test.npy"))

# Scale input to Q7.8 (Integer)
X_test_int = np.round(X_test * SCALE_FACTOR).astype(np.int32)

# --- 3. THE HARDWARE-LOGIC SIMULATOR ---
print("🚀 Simulating Hardware Inference...")
correct = 0

for i in range(len(X_test_int)):
    # Layer 1: MAC and Bias
    # In hardware: (Input * Weight) + (Bias << 8)
    z1 = np.dot(X_test_int[i], W1) + (b1.astype(np.int32) * SCALE_FACTOR)
    
    # Layer 1: ReLU
    a1 = np.maximum(0, z1)
    
    # Layer 1: Shift down to stay in 16-bit range (The "Pipeline Flush")
    a1 = a1 // SCALE_FACTOR
    
    # Layer 2: MAC and Bias
    z2 = np.dot(a1, W2.flatten()) + (b2.astype(np.int32) * SCALE_FACTOR)
    
    # Output: Sigmoid approximation
    prob = sigmoid_fixed(z2)
    y_pred = 1 if prob >= 0.5 else 0
    
    if y_pred == y_test[i]:
        correct += 1

# --- 4. FINAL RESULTS ---
accuracy = (correct / len(y_test)) * 100
print("\n" + "="*40)
print(f"  QUANTIZED ACCURACY: {accuracy:.2f}%")
print("="*40)

if accuracy > 95:
    print("✅ SUCCESS: Your hardware math is as smart as the AI!")
else:
    print("⚠️ WARNING: Accuracy drop. Check your scaling factors.")

📂 Loading data and quantized weights...
🚀 Simulating Hardware Inference...

  QUANTIZED ACCURACY: 98.82%
✅ SUCCESS: Your hardware math is as smart as the AI!
